(r-comparisons-two-period)=
# Two-period DiD in ModernDiD and R

This comparison measures how closely {func}`~moderndid.drdid`,
{func}`~moderndid.ipwdid`, and {func}`~moderndid.ordid` reproduce their
counterparts in R's `DRDID` package on the National Supported Work data. Each
pair shows the Python result, the R result, and the numerical difference
between their ATT estimates and standard errors.

The R calls run inside Python through `rpy2` and receive the same observations
and covariates as the Python fits.

```{admonition} What these comparisons check
:class: note

These are panel specifications with analytical standard errors. The improved
DR estimator, unnormalized IPW estimator, and outcome regression estimator have
different assumptions and can return different treatment effects. Here we compare
each estimator with its own R counterpart.
```

## Use the same data in both languages

{func}`~moderndid.load_nsw` provides earnings in 1975 and 1978, a treatment
indicator, and baseline characteristics for 16,417 people observed in both
years. The treatment indicator `experimental` equals one for 425 of them and
zero for the other 15,992. We keep the two-period panel and adjust for age,
education, race, marital status, and whether the participant completed high
school.

The section on {ref}`running the comparisons <r-comparisons-running>`
explains how to install the optional R environment and rerun this notebook.

In [1]:
import rpy2.robjects as ro

import moderndid as did

data = did.load_nsw().select(
    "id", "year", "experimental", "re", "age", "educ", "black", "married", "nodegree", "hisp"
)
formula = "~ age + educ + black + married + nodegree + hisp"

Error importing in API mode: ImportError("dlopen(/Users/jordandeklerk/moderndid/.venv/lib/python3.12/site-packages/_rinterface_cffi_api.abi3.so, 0x0002): Library not loaded: /Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib\n  Referenced from: <05EE76C0-33BF-3D4C-BC56-84005241BC88> /Users/jordandeklerk/moderndid/.venv/lib/python3.12/site-packages/_rinterface_cffi_api.abi3.so\n  Reason: tried: '/Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib' (no such file), '/Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib' (no such file)")
Trying to import in ABI mode.


In [2]:
import numpy as np
import pandas as pd
from rpy2.robjects import pandas2ri

with (ro.default_converter + pandas2ri.converter).context():
    ro.globalenv["nsw"] = ro.conversion.py2rpy(data.to_pandas())

## Improved doubly robust DiD

We set `est_method="imp"` in Python and `estMethod="imp"` in R. Both calls use
a panel, the same covariate formula, and analytical inference. The Python summary
appears immediately below its call.

In [3]:
python_dr = did.drdid(
    data=data, yname="re", tname="year", idname="id", treatname="experimental",
    xformla=formula, panel=True, est_method="imp", boot=False,
)
print(python_dr)

 Doubly Robust DiD Estimator (Improved Method)
 Computed from 32834 observations and 6 covariates.

┌───────────┬────────────┬──────────┬──────────────────────────┐
│       ATT │ Std. Error │ Pr(>|t|) │ [95% Conf. Interval]     │
├───────────┼────────────┼──────────┼──────────────────────────┤
│ -604.0295 │   405.8350 │   0.1367 │ [-1399.4662, 191.4072]   │
└───────────┴────────────┴──────────┴──────────────────────────┘

------------------------------------------------------------------------------
 Data Info
------------------------------------------------------------------------------
 Data structure: Panel data

------------------------------------------------------------------------------
 Estimation Details
------------------------------------------------------------------------------
 Outcome regression: Weighted least squares
 Propensity score: Inverse probability tilting

------------------------------------------------------------------------------
 Inference
----------------

The R fit uses `dname` for the treatment indicator. Its summary reports the ATT
and standard error from the same observations.

In [4]:
with ro.default_converter.context():
    r_dr = ro.r("""
    DRDID::drdid(
        data = nsw, yname = "re", tname = "year", idname = "id",
        dname = "experimental",
        xformla = ~ age + educ + black + married + nodegree + hisp,
        panel = TRUE, estMethod = "imp", boot = FALSE
    )
    """)
    _ = ro.r["summary"](r_dr)

 Call:
DRDID::drdid(yname = "re", tname = "year", idname = "id", dname = "experimental", 
    xformla = ~age + educ + black + married + nodegree + hisp, 
    data = nsw, panel = TRUE, estMethod = "imp", boot = FALSE)
------------------------------------------------------------------
 Further improved locally efficient DR DID estimator for the ATT:
 
   ATT     Std. Error  t value    Pr(>|t|)  [95% Conf. Interval] 
-604.0295   405.8351   -1.4884     0.1367   -1399.4664  191.4073 
------------------------------------------------------------------
 Estimator based on panel data.
 Outcome regression est. method: weighted least squares.
 Propensity score est. method: inverse prob. tilting.
 Analytical standard error.
------------------------------------------------------------------
 See Sant'Anna and Zhao (2020) for details.

Since the summaries round their numbers for display, we extract the underlying
ATT and standard error and subtract them to check agreement beyond that
rounding. The difference columns report Python minus R before any rounding.
The notebook fails if an ATT differs from R by more than
`1e-6 + 1e-6 * abs(R ATT)` or a standard error by more than
`1e-6 + 1e-4 * abs(R SE)`.

In [5]:
def compare_two_period(python_fit, r_fit, estimator):
    """Set a Python fit beside its R counterpart and check the tolerances."""
    with ro.default_converter.context():
        r_att = r_fit.rx2("ATT")[0]
        r_se = r_fit.rx2("se")[0]
    np.testing.assert_allclose(python_fit.att, r_att, rtol=1e-6, atol=1e-6)
    np.testing.assert_allclose(python_fit.se, r_se, rtol=1e-4, atol=1e-6)
    return pd.DataFrame({
        "estimator": [estimator],
        "Python ATT": [python_fit.att],
        "R ATT": [r_att],
        "ATT difference": [python_fit.att - r_att],
        "Python SE": [python_fit.se],
        "R SE": [r_se],
        "SE difference": [python_fit.se - r_se],
    })


compare_two_period(python_dr, r_dr, "drdid")

,estimator,Python ATT,R ATT,ATT difference,Python SE,R SE,SE difference
0,drdid,-604.029527,-604.029527,5.196625e-10,405.835041,405.835134,-0.000092


## Inverse probability weighting

Unnormalized IPW takes `est_method="ipw"` in Python and `normalized=FALSE` in R.
Since changing either setting changes the estimator, check it when you
translate an existing analysis. Both summaries below confirm the choice by
naming Horvitz-Thompson-type weights.

In [6]:
python_ipw = did.ipwdid(
    data=data, yname="re", tname="year", idname="id", treatname="experimental",
    xformla=formula, panel=True, est_method="ipw", boot=False,
)
print(python_ipw)

 Inverse Probability Weighted DiD Estimator
 Computed from 32834 observations and 6 covariates.

┌───────────┬────────────┬──────────┬──────────────────────────┐
│       ATT │ Std. Error │ Pr(>|t|) │ [95% Conf. Interval]     │
├───────────┼────────────┼──────────┼──────────────────────────┤
│ -813.3648 │   419.0902 │   0.0523 │ [-1634.7816,   8.0520]   │
└───────────┴────────────┴──────────┴──────────────────────────┘

------------------------------------------------------------------------------
 Data Info
------------------------------------------------------------------------------
 Data structure: Panel data

------------------------------------------------------------------------------
 Estimation Details
------------------------------------------------------------------------------
 Weight type: Unnormalized (Horvitz-Thompson-type)
 Propensity score: Logistic regression

------------------------------------------------------------------------------
 Inference
--------------------

In [7]:
with ro.default_converter.context():
    r_ipw = ro.r("""
    DRDID::ipwdid(
        data = nsw, yname = "re", tname = "year", idname = "id",
        dname = "experimental",
        xformla = ~ age + educ + black + married + nodegree + hisp,
        panel = TRUE, normalized = FALSE, boot = FALSE
    )
    """)
    _ = ro.r["summary"](r_ipw)

 Call:
DRDID::ipwdid(yname = "re", tname = "year", idname = "id", dname = "experimental", 
    xformla = ~age + educ + black + married + nodegree + hisp, 
    data = nsw, panel = TRUE, normalized = FALSE, boot = FALSE)
------------------------------------------------------------------
 IPW DID estimator for the ATT:
 
   ATT     Std. Error  t value    Pr(>|t|)  [95% Conf. Interval] 
-813.3648   419.0902   -1.9408     0.0523   -1634.7816   8.052   
------------------------------------------------------------------
 Estimator based on panel data.
 Horvitz-Thompson-type IPW estimator.
 Propensity score est. method: maximum likelihood.
 Analytical standard error.
------------------------------------------------------------------
 See Sant'Anna and Zhao (2020) for details.

In [8]:
compare_two_period(python_ipw, r_ipw, "ipwdid")

,estimator,Python ATT,R ATT,ATT difference,Python SE,R SE,SE difference
0,ipwdid,-813.364806,-813.364806,6.743903e-10,419.090189,419.090189,-6.679102e-11


## Outcome regression

The final pair uses outcome regression on the same sample and covariates. As
above, the estimates shown in the comparison come directly from these fits.

In [9]:
python_or = did.ordid(
    data=data, yname="re", tname="year", idname="id", treatname="experimental",
    xformla=formula, panel=True, boot=False,
)
print(python_or)

 Outcome Regression DiD Estimator
 Computed from 32834 observations and 6 covariates.

┌────────────┬────────────┬──────────┬───────────────────────────┐
│        ATT │ Std. Error │ Pr(>|t|) │ [95% Conf. Interval]      │
├────────────┼────────────┼──────────┼───────────────────────────┤
│ -1010.2766 │   365.1046 │   0.0057 │ [-1725.8816, -294.6717] * │
└────────────┴────────────┴──────────┴───────────────────────────┘

------------------------------------------------------------------------------
 Data Info
------------------------------------------------------------------------------
 Data structure: Panel data

------------------------------------------------------------------------------
 Estimation Details
------------------------------------------------------------------------------
 Estimation method: Ordinary least squares

------------------------------------------------------------------------------
 Inference
-------------------------------------------------------------------

In [10]:
with ro.default_converter.context():
    r_or = ro.r("""
    DRDID::ordid(
        data = nsw, yname = "re", tname = "year", idname = "id",
        dname = "experimental",
        xformla = ~ age + educ + black + married + nodegree + hisp,
        panel = TRUE, boot = FALSE
    )
    """)
    _ = ro.r["summary"](r_or)

 Call:
DRDID::ordid(yname = "re", tname = "year", idname = "id", dname = "experimental", 
    xformla = ~age + educ + black + married + nodegree + hisp, 
    data = nsw, panel = TRUE, boot = FALSE)
------------------------------------------------------------------
 Outcome-Regression DID estimator for the ATT:
 
   ATT     Std. Error  t value    Pr(>|t|)  [95% Conf. Interval] 
-1010.2766  365.1046   -2.7671     0.0057   -1725.8816 -294.6717 
------------------------------------------------------------------
 Estimator based on panel data.
 Outcome regression est. method: OLS.
 Analytical standard error.
------------------------------------------------------------------
 See Sant'Anna and Zhao (2020) for details.

In [11]:
compare_two_period(python_or, r_or, "ordid")

,estimator,Python ATT,R ATT,ATT difference,Python SE,R SE,SE difference
0,ordid,-1010.276624,-1010.276624,6.934897e-10,365.104563,365.104563,-5.871925e-11


The paired results show the size of the remaining numerical differences for each
specification. These checks cover the panel estimators shown here and do not
establish agreement for repeated cross-sections or bootstrap inference.

## Repeated cross-sections

With `panel=False` and covariates, the `"trad"` method of {func}`~moderndid.drdid`
and {func}`~moderndid.drdid_trad_rc` give the same estimates as `DRDID` and
differ by design in their standard errors. Since both estimators measure changes
from the pre-treatment period, the pre-treatment averages enter those changes
with a minus sign. The estimation effect of the comparison group's
pre-treatment regression reaches the influence function through those averages
and keeps their sign. `DRDID` 1.3.0 adds that effect instead (`R/drdid_rc.R`,
lines 331 and 332, and `R/drdid_rc1.R`, lines 195 and 220). A finite-difference
derivative of the estimate confirms the minus sign. `R/drdid_rc.R` uses that
sign itself for the matching term of its locally efficient adjustment at line
334. On the National Supported Work data treated as repeated cross-sections,
with age, education, race, and marital status as covariates, the `"trad"`
standard error is 492.32 in ModernDiD and 494.00 in `DRDID`.

## Designs the regressions can't identify

Unlike the National Supported Work data, some designs leave a regression
unidentified because one of its groups has fewer observations than
coefficients or collinear covariates. ModernDiD's two-period estimators then
stop with an error that names the group. The improved estimators of `DRDID` 1.3.0 still return a number by
setting the coefficients they cannot identify to zero.

:::{admonition} DRDID reports an undefined estimate
:class: warning

Since the number that `DRDID` returns can change when you reorder or rescale
the covariate columns, the method defines no such estimate. Rather than report
it, change the covariates as ModernDiD's error message suggests.
:::

## Versions used

The outputs above were produced with the following versions.

In [12]:
from importlib.metadata import version
from platform import python_version

with ro.default_converter.context():
    r_version = ro.r("R.version.string")[0]
    drdid_version = ro.r('as.character(packageVersion("DRDID"))')[0]
pd.DataFrame({
    "component": ["Python", "ModernDiD", "rpy2", "R", "DRDID"],
    "version": [python_version(), version("moderndid"), version("rpy2"), r_version, drdid_version],
})

,component,version
0,Python,3.12.10
1,ModernDiD,0.2.0
2,rpy2,3.6.4
3,R,R version 4.3.3 (2024-02-29)
4,DRDID,1.3.0
